In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Heart rate during exercise

**New concept: `ewm(halflife=N)`**

`halflife` is an alternative to `span` for controlling how fast older values decay:

```python
s.ewm(halflife=3).mean()   # after 3 periods, a past value's weight is halved
s.ewm(span=5).mean()       # span controls decay differently — smaller alpha, smoother
```

`halflife` is more intuitive when you can answer "how many periods until this data is half as relevant?" Small halflife = reacts quickly. Large halflife = long memory.

Rule of thumb: `span ≈ 2 × halflife` gives roughly similar smoothing, but they are not identical.

---

A runner's heart rate (bpm) recorded every minute during a 25-minute workout.

1. Add `ewm_span = bpm.ewm(span=6).mean()` and `ewm_half = bpm.ewm(halflife=3).mean()`. Print both alongside `bpm` for the first 10 rows — how do they differ early on?
2. `np.corrcoef` on `ewm_span` and `ewm_half` — how similar are they overall?
3. Which minute has the largest absolute difference between the two? Use `np.abs` and `np.argmax`.

In [12]:
hr = pd.DataFrame({
    'minute': range(1, 26),
    'bpm':    [ 72,  85,  98, 112, 108, 124, 131, 127, 138, 142,
               136, 144, 149, 155, 151, 158, 162, 156, 163, 168,
               172, 165, 158, 152, 144],
})

# Your code here


hr['ewm_span'] = hr['bpm'].ewm(span=6).mean()
hr['ewm_half'] = hr['bpm'].ewm(halflife=3).mean()


print(hr.iloc[:10][['bpm','ewm_span','ewm_half']])
print('similar')

c = np.corrcoef(hr['ewm_span'], hr['ewm_half'])[0,1]
print('correlation is: ',c)
print('very highly correlated')

print(hr.loc[np.argmax(np.abs(hr['ewm_span']- hr['ewm_half'])),'minute'],'has the largest absolute difference')



   bpm    ewm_span    ewm_half
0   72   72.000000   72.000000
1   85   79.583333   79.247587
2   98   87.862385   86.984813
3  112   97.185811   95.540930
4  108  100.981289   99.293084
5  124  108.565298  106.089116
6  131  115.647017  112.500373
7  127  119.126497  116.050795
8  138  124.793196  121.225777
9  142  129.885474  125.983516
similar
correlation is:  0.9989959752569963
very highly correlated
14 has the largest absolute difference


---

## Level 2 — Building energy consumption

Weekly electricity usage (kWh) for an office building over 30 weeks. A heatwave in week 16 caused a large spike.

This exercise puts `rolling`, `expanding`, and `ewm` side by side on the same data — use it to see how they differ.

1. Add `roll4 = kwh.rolling(4).mean()`, `exp_mean = kwh.expanding().mean()`, and `ewm5 = kwh.ewm(span=5).mean()`. Print rows 13–20 (the spike window) for all four columns.
2. By week 20 (4 weeks after the spike), which smoother has returned closest to the pre-spike baseline? Use `np.abs` to measure each smoother's distance from the week-15 value.
3. Use `np.argmax` on each of the three smoothers to find which week each peaked. Which smoother peaked latest?

In [ ]:
energy = pd.DataFrame({
    'week': pd.date_range('2022-01-03', periods=30, freq='W'),
    'kwh':  [4200, 4350, 4180, 4520, 4390, 4610, 4480, 4550, 4620, 4490,
             4700, 4650, 4820, 4760, 4900, 7800, 5100, 4850, 4780, 4920,
             4860, 5010, 4950, 5100, 5040, 5180, 5120, 5260, 5200, 5380],
})

# Your code here
energy['roll4'] = energy['kwh'].rolling(4).mean()
energy['exp_mean'] = energy['kwh'].expanding().mean()
energy['ewm5'] = energy['kwh'].ewm(span = 5).mean()

print(energy.iloc[13:21][['kwh','roll4','exp_mean','ewm5']])

a1 = energy.iloc[19]['roll4'] - energy.iloc[14]['kwh'] 
a2 = energy.iloc[19]['exp_mean'] - energy.iloc[14]['kwh'] 
a3 = energy.iloc[19]['ewm5'] - energy.iloc[14]['kwh'] 

m = ['roll4','exp_mean','ewm5']
r =[np.abs(a1),np.abs(a2),np.abs(a3)]

print(m[np.argmin(r)], 'is the closest')
p = {}
for i in m:
    p[i] = energy.loc[np.argmax(energy[i]),'week']

print(p)
print(max(p, key = p.get),'peaked latest')



     kwh   roll4     exp_mean         ewm5
13  4760  4732.5  4522.857143  4704.596125
14  4900  4782.5  4548.000000  4769.879836
15  7800  5570.0  4751.250000  5781.459959
16  5100  5640.0  4771.764706  5554.075854
17  4850  5662.5  4776.111111  5319.224993
18  4780  5632.5  4776.315789  5139.402212
19  4920  4912.5  4783.500000  5066.246141
20  4860  4852.5  4787.142857  4997.483642
ewm5 is the closest
{'roll4': Timestamp('2022-05-08 00:00:00'), 'exp_mean': Timestamp('2022-07-31 00:00:00'), 'ewm5': Timestamp('2022-04-24 00:00:00')}
exp_mean peaked latest


---

## Level 3 — Two customer segments

Monthly spend for `Premium` and `Standard` segments over 12 months.

1. Per segment: add `mom_growth` via `groupby().transform(pct_change)`. Which segment × month had the highest single-month growth? Add `cum_avg_growth` via `groupby().transform(expanding().mean())` on `mom_growth` — how does each segment's cumulative average growth rate evolve?
2. Per segment: add `ewm_spend` via `groupby().transform(ewm(span=4).mean())`. At the final month, what's the gap between the two segments' ewm trends?
3. Extract each segment's `spend` as a plain array. `np.corrcoef` — do the two segments move together?
4. `np.percentile` on `spend` across both segments combined — find Q3. Which segment has more months above it, and by how much?

In [75]:
segments = pd.DataFrame({
    'month':   list(pd.date_range('2023-01', periods=12, freq='MS')) * 2,
    'segment': ['Premium']*12 + ['Standard']*12,
    'spend':   [
         8200,  8650,  8400,  9100,  9480,  9250,
         9870, 10200,  9950, 10600, 11100, 11800,   # Premium: strong growth
        15400, 15800, 15200, 16100, 15900, 16400,
        15700, 16200, 15800, 16300, 15900, 16500,   # Standard: plateauing
    ],
})

# Your code here
segments['mom_growth'] = segments.groupby('segment')['spend'].transform(lambda x: x.pct_change())
print(segments.loc[segments['mom_growth'].idxmax(),'month'])
print(segments.loc[segments['mom_growth'].idxmax(),'segment'])
segments['cum_avg_growth'] = segments.groupby('segment')['mom_growth'].transform(lambda x: x.expanding().mean())

segments['ewm_spend'] = segments.groupby('segment')['spend'].transform(lambda x: x.ewm(span = 4).mean())

m = segments['month'].unique()[-1]

mm = segments.loc[segments['month'] == m]
pe = mm.loc[segments['segment'] == 'Premium','ewm_spend']
se = mm.loc[segments['segment'] == 'Standard','ewm_spend']

print(np.array(pe) - np.array(se))

sp = [s for s in segments.loc[segments['segment'] == 'Premium','spend']]
ss = [s for s in segments.loc[segments['segment'] == 'Standard','spend']]
cc = np.corrcoef(sp, ss)[0,1]
print(f'correlation is {cc:.2f}')


qs = np.percentile(segments['spend'], q = [25,75])
g = segments.groupby('segment')['spend'].apply(lambda x: (x>qs[1]).sum())

print(g)

print(g.idxmax(), 'has more')
ag = np.array(g)
print('by', ag[1]- ag[0])


2023-04-01 00:00:00
Premium
[-5169.3751119]
correlation is 0.66
segment
Premium     0
Standard    5
Name: spend, dtype: int64
Standard has more
by 5
